# **Download Essential Libraries**

In [1]:
!pip install -q pypdf sentence-transformers scikit-learn transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 5.8 MB/s eta 0:00:00


# **Load the doc**

In [2]:
from google.colab import files
uploaded = files.upload()

path = list(uploaded.keys())[0]
print('Path to the doc:', path)

Saving Phishing.pdf to Phishing.pdf
Path to the doc: Phishing.pdf


# **Extract information from doc**

In [7]:
from pypdf import PdfReader

reader = PdfReader(path)
text = ''

for page in reader.pages:
  page_content = page.extract_text()

  if page_content:
    text += page_content + "\n"

print("No of characters:", len(text))
print("No of pages:", len(reader.pages))

No of characters: 35694
No of pages: 9


# **Create Chunks of doc**

In [8]:
def create_chunk(text, chunk_size = 500, overlap = 100):

  chunk = []
  start = 0

  while start < len(text):
    end = start + chunk_size
    chunk.append(text[start:end])

    start = end - overlap
  return chunk

chunks = create_chunk(text)

print("No of chunks:", len(chunks))

No of chunks: 90


In [9]:
for i, chunk in enumerate(chunks):
  print(f"\n-----Chunk {i+1}-----")
  print(chunk)


-----Chunk 1-----
EXPLICATE: Enhancing Phishing Detection through Explainable AI and
LLM-Powered Interpretability
Bryan Lim, Roman Huerta, Alejandro Sotelo, Anthonie Quintela, Priyanka Kumar
Department of Computer Science
University of Texas at Permian Basin
Odessa, Texas, USA
{lim p65274, huerta r71882, sotelo a92823, quintela a66563, kumar p}@utpb.edu
Abstract—Sophisticated phishing attacks have emerged as a
major cybersecurity threat, becoming more common and difficult
to prevent. Though machine learning techn

-----Chunk 2-----
r cybersecurity threat, becoming more common and difficult
to prevent. Though machine learning techniques have shown
promise in detecting phishing attacks, they function mainly as
”black boxes” without revealing their decision-making rationale.
This lack of transparency erodes the trust of users and diminishes
their effective threat response. We present EXPLICATE: a
framework that enhances phishing detection through a three-
component architecture: an ML-ba

# **Convert chunks into embeddings**

In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(chunks, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [11]:
print('Embedding Shape:', embeddings.shape)

Embedding Shape: (90, 384)


# **Ask a question**

In [20]:
question = input("Ask a question about the document:")

question_embedding = embedding_model.encode(question)

Ask a question about the document:wHEN WAS THE PAPER PUBLISHED


# **Find the most relevant chunks**

In [24]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarities = cosine_similarity(question_embedding.reshape(1, -1), embeddings)[0]

top_k = 3

top_indices = np.argsort(similarities)[-top_k:][::-1]

# **Display retrieved information**

In [25]:
print("\n===== RETRIEVED CONTEXT =====\n")

for rank, index in enumerate(top_indices):

    print(f"Rank {rank + 1}")
    print(f"Similarity: {similarities[index]:.4f}")
    print(chunks[index])
    print("\n" + "=" * 80)


===== RETRIEVED CONTEXT =====

Rank 1
Similarity: 0.0680
Further analysis of email characteristics revealed interesting
patterns in text length distribution, as shown in Fig. 7:
Fig. 7. Email length distribution by label (note logarithmic scale) showing
concentration of both classes in shorter lengths.
The length distribution analysis reveals that:
• Most emails (both legitimate and phishing) are concen-
trated in the shorter length ranges
• The distribution follows a logarithmic pattern, with fre-
quency decreasing as length increases
• Legitimate em

Rank 2
Similarity: 0.0441
ng, and real-time email content manipu-
lation to avoid detection by service providers. Traditional
machine learning technologies operate mainly on pre-
learnt models past models, and they were employed on
datasets to avoid being phished.
3) Many false negatives and false positives happen when
genuine emails are wrongly classified as phishing, which
disturbs business communications. On the other hand,
when user

# **Generate the answer**

In [27]:
!pip install -q huggingface_hub

In [30]:
from transformers import pipeline

generator = pipeline("text-generation", model = 'google/flan-t5-base')

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

# **Build RAG Prompt**

In [41]:
context = "\n\n".join(
    chunks[index]
    for index in top_indices
)

prompt = f"Answer the question: {question} based on the following context: {context}"

# **Generate Answer**

In [42]:
result = generator(prompt, max_new_tokens = 200)

answer = result[0]['generated_text']

print("\n===== ANSWER =====\n")
print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== ANSWER =====

Answer the question: wHEN WAS THE PAPER PUBLISHED based on the following context: Further analysis of email characteristics revealed interesting
patterns in text length distribution, as shown in Fig. 7:
Fig. 7. Email length distribution by label (note logarithmic scale) showing
concentration of both classes in shorter lengths.
The length distribution analysis reveals that:
• Most emails (both legitimate and phishing) are concen-
trated in the shorter length ranges
• The distribution follows a logarithmic pattern, with fre-
quency decreasing as length increases
• Legitimate em

ng, and real-time email content manipu-
lation to avoid detection by service providers. Traditional
machine learning technologies operate mainly on pre-
learnt models past models, and they were employed on
datasets to avoid being phished.
3) Many false negatives and false positives happen when
genuine emails are wrongly classified as phishing, which
disturbs business communications. On the ot